In [0]:
df_ember_intensity_bronze = spark.read.table("portfolio_energia.bronze.ember_intensity")
df_ember_demand_bronze = spark.read.table("portfolio_energia.bronze.ember_demand")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


def silver_treatment(df_start):
    janela_deduplicacao = Window.partitionBy("iso3", "date").orderBy(col("data_ingestao_bronze").desc())
    
    df_final = (
        df_start
        .dropna("any", subset=["iso3", "date"]) # Limpa chaves nulas antes de agrupar na janela
        .withColumn("numero_linha", row_number().over(janela_deduplicacao))
        .filter(col("numero_linha") == 1) # Mantém apenas a linha mais recente!
        .drop("numero_linha")
        .withColumn("country", trim(col("country")))
        .withColumn("iso3", upper(trim(col("iso3"))))
        .withColumn("date", to_date(col("date")))
        .withColumn("value", col("value").cast("double"))
        .withColumn("metric_name", trim(col("metric_name")))
        .withColumn("data_ingestao_silver", current_timestamp())
        .fillna(-1.0, subset=["value"]) 
    )
    return df_final

def merge_silver_incremental(df_tratado, nome_tabela_silver):
    caminho_tabela = f"portfolio_energia.silver.{nome_tabela_silver}"

    # Se a tabela Silver ainda não existir (1ª execução), cria ela do zero
    if not spark.catalog.tableExists(caminho_tabela):
        df_tratado.write.format("delta").saveAsTable(caminho_tabela)
        
    else:
        df_tratado.createOrReplaceTempView("vw_dados_novos")
        spark.sql(f"""
            MERGE INTO {caminho_tabela} AS destino
            USING vw_dados_novos AS origem
            ON destino.iso3 = origem.iso3 AND destino.date = origem.date
            
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

df_ember_intensity_clean = silver_treatment(df_ember_intensity_bronze)
df_ember_demand_clean = silver_treatment(df_ember_demand_bronze)

merge_silver_incremental(df_ember_intensity_clean, "ember_intensity_silver")
merge_silver_incremental(df_ember_demand_clean, "ember_demand_silver")